# 베이스라인(XGBoost, TabPFN v3.5) vs Group-wise Cross-Attention vs 전체 데이터 한번에

| feature 세트 | 내용 |
|---|---|
| `subset` | 공선성 축소 + 그룹별 파생변수 (`src.features.build`, cross-attention 실험과 동일) |
| `full_raw` | 원본 feature 284개 전체 + 요일 (가공 없음, "전체 데이터 한번에") |
| `+exaone` | EXAONE Finance 1.0 zero-shot rolling 예측 feature 추가 (`run_exaone_feats.py`; FI·SH1·KE 80개 계열의 3일 뒤 예측 변화량과 불확실성) |

- 모델 표기는 `모델[feature 세트]`입니다. 신경망 모델은 `run_xattn.py`(subset)와 `run_xattn.py --full`(full_raw)로 실행합니다. TabPFN은 `run_tabpfn.py`로 **별도 프로세스**에서 실행합니다. XGBoost와 같은 커널에서 돌리면 libomp 충돌로 데드락이 나기 때문입니다.
- split: stratified, random, time(+3일 purge)이고 모든 모델이 같은 split을 씁니다. **stratified와 random split은 인접일 자기상관 때문에 낙관적인 점수가 나오므로, test와 가장 가까운 지표는 time split입니다.**

In [ ]:
import os, json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve
from xgboost import XGBClassifier
from src.data import ROOT, SCHEMES, TARGETS, group_columns, inner_split, load, make_split
from src.features import build, featsets

df, is_train = load()
groups = group_columns(df)
_, _, report = build(df, is_train, groups)
sets = featsets(df, is_train, groups)
y = df.loc[is_train, TARGETS].to_numpy().astype(int)
print({k: v.shape[1] for k, v in sets.items()})

## 공선성 리포트 요약

In [ ]:
pd.DataFrame({g: {k: v for k, v in r.items() if k != "clusters"} for g, r in report["within"].items()}).T

In [ ]:
display(pd.DataFrame(report["cca_level"]).round(2).style.set_caption("CCA (level)"))
display(pd.DataFrame(report["cca_diff7"]).round(2).style.set_caption("CCA (7일 차분)"))
pd.DataFrame({t: {g: r["top5_mean"] for g, r in v.items()} for t, v in report["target_relevance"].items()}).round(3)

## XGBoost (feature 세트별, seed 5개 평균, inner split early stopping)

In [ ]:
SEEDS = 5
def run_xgb(X, tr, va, es, t):
    ps = []
    for s in range(SEEDS):
        m = XGBClassifier(n_estimators=1000, learning_rate=0.03, max_depth=3, subsample=0.8, colsample_bytree=0.5,
                          min_child_weight=5, reg_lambda=1.0, eval_metric="auc", early_stopping_rounds=50, random_state=s)
        m.fit(X[tr], y[tr, t], eval_set=[(X[es], y[es, t])], verbose=False)
        ps.append(m.predict_proba(X[va])[:, 1])
    return np.mean(ps, 0)

rows, preds = [], {}
for fs, F in sets.items():
    X = F[is_train].to_numpy(np.float32)
    for scheme in SCHEMES:
        tr, va = make_split(y, scheme, seed=0)
        tr_in, es = inner_split(y, tr, scheme)
        for ti, t in enumerate(TARGETS):
            p = run_xgb(X, tr_in, va, es, ti)
            preds[f"{scheme}|{t}|xgboost[{fs}]"] = {"y": y[va, ti], "p": p}
            rows.append(dict(scheme=scheme, target=t, model=f"xgboost[{fs}]", auc=roc_auc_score(y[va, ti], p)))
pd.DataFrame(rows).pivot(index=["scheme", "target"], columns="model", values="auc").round(3)

## TabPFN v3.5 (`run_tabpfn.py` 결과 로드)

In [ ]:
tp_path = ROOT / "results/tabpfn_results.json"
if tp_path.exists():
    tp = json.load(open(tp_path))
    print("TabPFN config:", tp["config"])
    for r in tp["runs"]:
        rows.append(dict(scheme=r["scheme"], target=r["target"], model=f"tabpfn[{r['featset']}]", auc=r["auc"]))
    for k, v in tp["val_preds"].items():
        preds[k] = {"y": np.array(v["y"]), "p": np.array(v["p"])}
else:
    print("results/tabpfn_results.json 없음 → `uv run python run_tabpfn.py` 먼저 실행")

## 신경망 결과 로드 (subset: cross-attention 계열, full_raw: 단일 스트림)

In [ ]:
for fname, fs in [("xattn_results.json", "subset"), ("xattn_full_results.json", "full_raw")]:
    p = ROOT / "results" / fname
    if not p.exists():
        print(fname, "없음"); continue
    res = json.load(open(p))
    print(fname, "Q/KV 배정:", res["assignment"] or "단일 스트림")
    for r in res["runs"]:
        rows.append(dict(scheme=r["scheme"], target=r["target"], model=f"{r['variant']}[{fs}]", auc=r["auc_tta"]))
        if r["variant"] == "xattn":
            rows.append(dict(scheme=r["scheme"], target=r["target"], model=f"xattn_kv_shuffle[{fs}]", auc=r["auc_kv_shuffle"]))
    for k, v in res["val_preds"].items():
        s, t, var = k.split("|")
        preds[f"{s}|{t}|{var}[{fs}]"] = {"y": np.array(v["y"]), "p": np.array(v["p"])}

## 전체 비교표

In [ ]:
comp = pd.DataFrame(rows).pivot_table(index=["scheme", "target"], columns="model", values="auc")
time_tab = comp.xs("time", level="scheme").T.sort_values("Target_1", ascending=False)
print("== time split (test와 가장 가까운 지표), Target_1 기준 정렬")
display(time_tab.round(3))
comp.T.round(3)

### feature 세트 효과 (time split)
같은 모델 안에서 feature 세트만 바꿨을 때의 AUC입니다. 서브셋 구조, 전체 데이터, EXAONE feature 추가의 효과를 이 표로 분리해서 볼 수 있습니다.

In [ ]:
t = time_tab.copy()
t["base"] = t.index.str.split("[").str[0]
t["featset"] = t.index.str.extract(r"\[(.*)\]", expand=False).values
t.pivot_table(index=["base"], columns="featset", values=TARGETS).round(3)

### cross-attention 진단 (subset)
- **gate**: tanh(gate) 값입니다. 0에 가까우면 cross attention이 출력에 거의 기여하지 않습니다.
- **attn_entropy**: 1이면 attention이 균등 분포입니다.
- **kv_effect**: KV를 셔플했을 때 떨어지는 AUC입니다. 0 근처면 모델이 KV를 쓰지 않습니다.

In [ ]:
res = json.load(open(ROOT / "results/xattn_results.json"))
diag = pd.DataFrame([r for r in res["runs"] if r["variant"] == "xattn"])
diag = diag.assign(kv_effect=diag.auc_plain - diag.auc_kv_shuffle)
diag[["scheme", "target", "auc_plain", "auc_kv_shuffle", "kv_effect", "gate", "attn_entropy", "ffn_depth", "auc_seed_std"]].round(3)

### split별 낙관 편향 (shuffle 계열 split AUC − time split AUC)

In [ ]:
gap = comp.xs("time", level="scheme")
pd.concat({s: comp.xs(s, level="scheme") - gap for s in ["stratified", "random"]}).T.round(3)

## ROC 곡선 (time split, 상위 모델)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for ax, t in zip(axes, TARGETS):
    top = time_tab[t].sort_values(ascending=False).index[:6]
    for v in top:
        k = f"time|{t}|{v}"
        if k in preds:
            fpr, tpr, _ = roc_curve(preds[k]["y"], preds[k]["p"])
            ax.plot(fpr, tpr, label=f"{v} ({roc_auc_score(preds[k]['y'], preds[k]['p']):.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=0.8)
    ax.set(title=f"{t} — time split", xlabel="FPR", ylabel="TPR")
    ax.legend(fontsize=7)
plt.tight_layout()